# 01 — Learn LangChain (via the persona council)

Goal: understand the *one-LLM-call* building blocks before we wire them into a graph in
notebook 02. We learn them on the real thing — a council persona.

**Three concepts:**
1. `ChatPromptTemplate` — a reusable prompt with `{variables}`.
2. `ChatAnthropic` — the Claude model wrapper.
3. `.with_structured_output(Model)` — force a typed, parsed result (no string wrangling).

> Needs `ANTHROPIC_API_KEY` in your environment for the live cells. The offline cell runs
> with a fake model so you can see the wiring without a key.

## The structured-output schema

We make the LLM return *this* shape, not free text. LangChain validates and parses it for us.

In [ ]:
from quantwheel.agents.personas.base import PersonaOpinion, format_evidence
PersonaOpinion.model_json_schema()["properties"].keys()

## A persona = prompt | model.with_structured_output(...)

That single LCEL pipe IS the agent. Subclasses only change the system prompt.

In [ ]:
# OFFLINE: inject a fake model so you can see a vote without an API key.
from langchain_core.runnables import RunnableLambda
from quantwheel.agents.personas.base import PersonaAgent

fake = RunnableLambda(lambda _: PersonaOpinion(
    stance="FOR", conviction=0.7, rationale="liquid, positive EV", key_points=["EV>0", "liquid"]))
demo = PersonaAgent("Demo", "You are a demo voter.", llm=fake)
# (build a candidate from a live scan, or use the test fixture's shape)
print("chain =", type(demo._runnable()).__name__)

In [ ]:
# LIVE: a real persona voting on a real candidate (needs ANTHROPIC_API_KEY).
from quantwheel.services.scan_service import run_weekly_scan
from quantwheel.agents.council import run_council

scan = run_weekly_scan(["NIO", "F"])
verdicts = run_council(scan.candidates)   # 5 personas each
for v in verdicts:
    print(f"\n=== {v.ticker}  net {v.aggregate_score:+.2f} ===")
    print(v.consensus)
    for vote in v.votes:
        print(f"  {vote.persona:16} {vote.stance.value:14} ({vote.conviction:.0%}) {vote.rationale}")

## Going faster: `.batch()`

Each persona is a Runnable, so `agent._runnable().batch([...])` evaluates many candidates
concurrently. For a weekly one-position account the loop is plenty, but batching is how you'd
scale the council to a large universe.

Next → **02_learn_langgraph.ipynb**: wrap scan + council + gates into a stateful weekly agent
with a human-approval step.